## Practical 9: Introduction to Image Classification Using DNNs

The objectives of this practical are:

- To understand the basics of convolutional neural networks
- To learn how to build CNNs
- To learn the workflow of working with a CNN model for an ML application
- To get hands-on expereince in using the relevant functions in the workflow of a CNN in the relevant libraries, i.e. tensorflow and keras
- To run the notebook on both jupyterhub and Google Colab

In this practical we will first build an artificial neural network (ANN) in Keras to classify the fashion_mnist dataset provided by Keras. This will act as a baseline for comparison with the CNN models that we will build next.

We will then build two alternative CNN models with different architectures to classify the same dataset. 

The fashion_mnist dataset contains 70,000 fasion images with 10 categories, with 60,000 for training and 10,000 for testing. Each image is represented as 28x28 gray-scale pixels.

The 10 classes are labelled as follows:

- 0 T-shirt/top 
- 1 Trouser 
- 2 Pullover 
- 3 Dress 
- 4 Coat 
- 5 Sandal 
- 6 Shirt 
- 7 Sneaker 
- 8 Bag 
- 9 Ankle boot

We will show the performances by these different models.

For further information on Keras, refere to: 

https://keras.io/

## Tutorial 1: Building Artificial Neural Networks in Keras


## Setup

In [ ]:
from tensorflow import keras
from keras import layers

In [ ]:
from numpy.random import seed
seed(0)
from tensorflow import random 
random.set_seed(0) # Sets the global random seed.


### Import the fasion_mnist dataset

We split the dataset into the training and test datasets.

In [ ]:
from keras.datasets import fashion_mnist
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()



Let's have a quick look at the shapes of the datasets.

In [ ]:
print('x_train shape:', x_train.shape)
print('y_train shape:', y_train.shape)
print('x_test shape:', x_test.shape)
print('y_test shape:', y_test.shape)

import numpy as np
num_classes = len(np.unique(y_train))
print('num classes:', num_classes)
print('class labels:', np.unique(y_train))


### Visualise some of the images/instances.

In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(8,8))

for i in range(num_classes):
  fig.add_subplot(2, 5, i+1)

  idx = np.where(y_train==i)[0] # one image for each class
  im = x_train[idx][0]

  # plt.imshow(im, plt.get_cmap('gray')) # Show one image with cmap
  plt.imshow(im) # without cmap


### Data preprocessing

In [ ]:
print('x_train shape:', x_train.shape)

x_train_flat = x_train.reshape(x_train.shape[0], x_train.shape[1]*x_train.shape[2])
print('x_train_flat shape:', x_train_flat.shape)

x_test_flat = x_test.reshape(x_test.shape[0], x_test.shape[1]*x_test.shape[2])
print('x_test_flat shape:', x_test_flat.shape)

print('example y class:', y_train[3])

y_train_categorical = keras.utils.to_categorical(y_train)
y_test_categorical = keras.utils.to_categorical(y_test)

print('example one hot-encoded class: ', y_train_categorical[3])
print('y_train_categorical shape: ', y_train_categorical.shape)

In [ ]:
from keras.models import Sequential
from keras.layers import Dense

### Building a fully connected ANN

For further information about the Sequential class, refer to:

https://keras.io/api/models/sequential/

In [ ]:
from keras.models import Sequential
from keras.layers import Dense

model = Sequential()

num_features = x_train_flat.shape[1]
model.add(Dense(128, input_dim = num_features, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(num_classes, activation='softmax'))

### Compile the model

In [ ]:
model.compile(loss = 'categorical_crossentropy', optimizer = 'sgd', metrics = ['accuracy'])

### Train the model

In [ ]:
history = model.fit(x_train_flat, y_train_categorical, epochs = 10) 

### Evaluate the model

In [ ]:
loss, accuracy = model.evaluate(x_test_flat, y_test_categorical)
print('loss :', loss, 'accuracy:', accuracy)

### Show loss versus epochs

In [ ]:
plt.plot(history.history['loss'])
plt.xlabel('epoch')
plt.ylabel('loss')
plt.show()

### Show accuracy versus epochs

In [ ]:
plt.plot(history.history['accuracy'])
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.show()

#### The performance is apparently not good. 


## Tutorial 2: Building CNN for image classification in Keras

We will use some of the Keras APIs:
- Conv2D() for creating a convolutional layer:https://keras.io/api/layers/convolution_layers/convolution2d/

- Pooling() for creating a pooling layer: https://keras.io/layers/pooling/

- Dropout() for applying drop out: https://keras.io/api/layers/regularization_layers/dropout/

- BatchNormalization(): https://keras.io/api/layers/convolution_layers/convolution2d/

In [ ]:
from keras.layers.convolutional import Conv2D
from keras.layers.convolutional import MaxPooling2D
from keras.layers import Flatten

In [ ]:
height = x_train.shape[1]
width = x_train.shape[2]
channels = 1

print(x_train.shape)
x_train_cnn = x_train.reshape(x_train.shape[0], height, width, channels)
x_test_cnn = x_test.reshape(x_test.shape[0], height, width, channels)

print(x_train_cnn.shape)


### Normalise the datasets

Convert the data into 32 bit float, making the data fit into the RAM better for training.

In [ ]:
x_train = x_train_cnn.astype('float32') / 255
x_test = x_test_cnn.astype('float32') / 255

### Build a simple CNN model

In [ ]:
model = Sequential()
model.add(Conv2D(filters=32, kernel_size=(5,5), padding='same', activation='relu', input_shape=(height, width, channels)))
model.add(MaxPooling2D(pool_size=(2,2), strides=(2,2)))

model.add(Conv2D(filters=64, kernel_size=(5,5), padding='same', activation='relu', ))
model.add(MaxPooling2D(pool_size=(2,2), strides=(2,2)))

model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(num_classes, activation='Softmax'))

# produce the model summary
model.summary()

### Compile the model

Configure the training process with the compile() before training the model with three arguments, loss, optimizer and metrics, which can be changed to create different models. 

In [ ]:
# model.compile(loss = 'categorical_crossentropy', optimizer = 'sgd', metrics = ['accuracy']) # can try this optimizer

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy']) # a different optimizer

### Train the model
No batch size is explicitly aspecified so the default value of 32 is used. 

Note that it takes a while to train the model.

In [ ]:
history = model.fit(x_train, y_train_categorical, epochs = 10)

### Evaluation with loss and accuracy

In [ ]:
loss, accuracy = model.evaluate(x_test, y_test_categorical)
print('loss :', loss, 'accuracy:', accuracy)

### Show loss versus epochs

In [ ]:
plt.plot(history.history['loss'])
plt.xlabel('epoch')
plt.ylabel('loss')
plt.show()

### Show accuracy versus epochs

In [ ]:
plt.plot(history.history['accuracy'])
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.show()

### This simple model has achieved very impressive performance.

### Build another CNN model with different layers

The architecture used in this model and the cells for validation are adapted from:

https://github.com/margaretmz/deep-learning/blob/master/fashion_mnist_keras.ipynb


In [ ]:
model = keras.Sequential()

# Must define the input shape in the first layer of the neural network
model.add(Conv2D(filters=64, kernel_size=2, padding='same', activation='relu', input_shape=(28,28,1))) 
model.add(MaxPooling2D(pool_size=2))
model.add(keras.layers.Dropout(0.3))

model.add(Conv2D(filters=32, kernel_size=2, padding='same', activation='relu'))
model.add(MaxPooling2D(pool_size=2))
model.add(keras.layers.Dropout(0.3))

model.add(Flatten())

model.add(Dense(256, activation='relu'))
model.add(keras.layers.Dropout(0.5))
model.add(Dense(10, activation='softmax'))

# Show the model summary
model.summary()

### Compile the model

Configure the training process with the compile() before training the model with three arguments, loss, optimizer and metrics, which can be changed to create different models. 

In [ ]:
# model.compile(loss = 'categorical_crossentropy', optimizer = 'sgd', metrics = ['accuracy'])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy']) # a different optimizer

### Further split the training dataset into the training and validation datasets

In [ ]:
# Further break training data into train / validation sets (# put 5000 into validation set and keep remaining 55,000 for train)
(x_train, x_valid) = x_train[5000:], x_train[:5000] 
(y_train, y_valid) = y_train[5000:], y_train[:5000]

# Reshape input data from (28, 28) to (28, 28, 1)
w, h = 28, 28
x_train = x_train.reshape(x_train.shape[0], w, h, 1)
x_valid = x_valid.reshape(x_valid.shape[0], w, h, 1)
x_test = x_test.reshape(x_test.shape[0], w, h, 1)

# One-hot encode the labels
y_train = keras.utils.to_categorical(y_train, 10)
y_valid = keras.utils.to_categorical(y_valid, 10)
y_test = keras.utils.to_categorical(y_test, 10)

# Print training set shape
print("x_train shape:", x_train.shape, "y_train shape:", y_train.shape)

# Print the number of training, validation, and test datasets
print(x_train.shape[0], 'train set')
print(x_valid.shape[0], 'validation set')
print(x_test.shape[0], 'test set')

### Training with validation

Note that it takes less time to train the model.

In [ ]:
from keras.callbacks import ModelCheckpoint

checkpointer = ModelCheckpoint(filepath='./weights.hdf5', verbose = 1, save_best_only=True)
history = model.fit(x_train,
         y_train,
         batch_size=64,
         epochs=10,
         validation_data=(x_valid, y_valid),
         callbacks=[checkpointer])

### Load Model with the best validation accuracy

In [ ]:
model.load_weights('./weights.hdf5')

### Evaluation with loss and accuracy 

In [ ]:
loss, accuracy = model.evaluate(x_test, y_test_categorical)
print('loss :', loss, 'accuracy:', accuracy)

### Show loss versus epochs

In [ ]:
plt.plot(history.history['loss'])
plt.xlabel('epoch')
plt.ylabel('loss')
plt.show()

### Show accuracy versus epochs

In [ ]:
plt.plot(history.history['accuracy'])
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.show()

#### This is a more complex model with more layers in the model and the validation dataset to tune the hyper-parameters. However, the model has not achieved better performance than the first model though it takes less time to train.

## Use the fitted mode to make predictions

### Make and visualise predictions on 15 images from the test dataset

First, we make the predictions with the model on the test dataset. Second, we randomly print out 15 images from the test dataset together with their predictions (and the groud truth labels). The correct predictions are in green while and the incorrect predictions are in red. 

In [ ]:
# Define the text class labels
fashion_mnist_labels = ["T-shirt/top",  
                        "Trouser",      
                        "Pullover",     
                        "Dress",        
                        "Coat",         
                        "Sandal",       
                        "Shirt",        
                        "Sneaker",      
                        "Bag",          
                        "Ankle boot"]

y_hat = model.predict(x_test)

# Plot a random sample of 10 test images, their predicted labels and ground truth
figure = plt.figure(figsize=(20, 8))
for i, index in enumerate(np.random.choice(x_test.shape[0], size=15, replace=False)):
    ax = figure.add_subplot(3, 5, i + 1, xticks=[], yticks=[])
    # Display each image
    ax.imshow(np.squeeze(x_test[index]))
    predict_index = np.argmax(y_hat[index])
    true_index = np.argmax(y_test[index])
    # Set the title for each image
    ax.set_title("{} ({})".format(fashion_mnist_labels[predict_index], 
                                  fashion_mnist_labels[true_index]),
                                  color=("green" if predict_index == true_index else "red"))

## Exercise 1: Compare the two different CNN architectures and identify the similar layers and different/additional layers.

## Exercise 2: Build and evaluate a CNN model to classify the digit images in the MNIST dataset (28x28)